# Notebook 9 — Inferência Econométrica de Desempenho e Caracterização do Benchmark

**TCC — Pedro Augusto Pinheiro Reis · Ciências Contábeis · UFG**

Este notebook operacionaliza a caracterização econométrica do benchmark (IBOVESPA) e executa os testes de hipóteses estatísticas comparando as carteiras ótimas contra o mercado.

## 1. Importações e Configurações

In [ ]:
import sys
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import het_arch
from arch import arch_model
from arch.unitroot import VarianceRatio

from utils.config_loader import carregar_parametros
from utils.inferencia import (
    fmt_pvalor,
    sharpe,
    sortino,
    sharpe_de_excesso,
    sortino_de_excesso,
    cagr,
    max_drawdown,
    _wald_spanning,
    _jk_memmel,
    bootstrap_ic,
    diagnostico_residuos,
    TRADING_DAYS
)

warnings.filterwarnings("ignore")

cfg = carregar_parametros()
SEED = cfg.get("SEED", 42)
BOOTSTRAP_REPS = cfg.get("BOOTSTRAP_REPS", 1000)
BOOTSTRAP_BLOCK_MEAN = cfg.get("BOOTSTRAP_BLOCK_MEAN", 21)
SPANNING_MAX_LAGS = cfg.get("SPANNING_MAX_LAGS", 5)

project_root = Path.cwd().parent.parent
INPUT_DIR_IBOV = project_root / "data" / "Ibov" / "Ibov_Diario"
INPUT_DIR_CDI = project_root / "data" / "CDI"
INPUT_DIR_STRATEGY = project_root / "data" / "Estrategias"
OUTPUT_DIR = project_root / "data" / "Estrategias"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ESTRATEGIAS = [
    "EqualWeight", "InvVol", "MinVar", "MinVar_c10", "MaxSharpe", "MaxSharpe_c10",
    "MaxOmega", "MaxSortino", "MaxKappa3", "MinCVaR", "MinCDaR"
]

print(f"✓ Configurações carregadas. Total de estratégias: {len(ESTRATEGIAS)}")

## 2. Ingestão de Dados e Alinhamento do Benchmark

In [ ]:
ibov_path = INPUT_DIR_IBOV / "ibov_diario_2010_2026.csv"
if not ibov_path.exists():
    raise FileNotFoundError("ibov_diario_2010_2026.csv não encontrado.")
ibov = pd.read_csv(ibov_path, parse_dates=["data"])
ibov = ibov.dropna(subset=["ibov_ret_simples"]).reset_index(drop=True)

cdi_path = INPUT_DIR_CDI / "cdi_diario_bcb_2010_atual.csv"
if not cdi_path.exists():
    raise FileNotFoundError("cdi_diario_bcb_2010_atual.csv não encontrado.")
cdi_df = pd.read_csv(cdi_path, sep=";", parse_dates=["data"])

cdi_serie = cdi_df.set_index("data")["cdi_diario"].reindex(ibov["data"]).ffill().bfill()
ibov["cdi_diario"] = cdi_serie.values
ibov["ibov_ret_log"] = np.log(1 + ibov["ibov_ret_simples"])

print(f"✓ Benchmark carregado e alinhado: {len(ibov):,} observações.")

## 3. Testes de Raiz Unitária e Ordem de Integração do IBOV

In [ ]:
r = ibov["ibov_ret_log"].dropna()
p = ibov["ibov_close"]

adf_r = adfuller(r, autolag="AIC")
adf_p = adfuller(p, autolag="AIC")
kpss_r = kpss(r, regression="c", nlags="auto")
kpss_p = kpss(p, regression="c", nlags="auto")

print("========================================================================")
print("TESTES DE ESTACIONARIEDADE DO BENCHMARK (IBOV)")
print("========================================================================\n")
print(f"ADF em log-retornos:   estat={adf_r[0]:.4f} | p-valor={fmt_pvalor(adf_r[1])}")
print(f"KPSS em log-retornos:  estat={kpss_r[0]:.4f} | p-valor={fmt_pvalor(kpss_r[1])}")
print(f"ADF em nível (close):  estat={adf_p[0]:.4f} | p-valor={fmt_pvalor(adf_p[1])}")
print(f"KPSS em nível (close): estat={kpss_p[0]:.4f} | p-valor={fmt_pvalor(kpss_p[1])}")

## 4. Diagnóstico de Momentos Superiores e Dependência Temporal

In [ ]:
arch_stat, arch_p, _, _ = het_arch(r, nlags=10)
jb_stat, jb_p = stats.jarque_bera(r)

print("========================================================================")
print("DIAGNÓSTICO ESTATÍSTICO DE COMPORTAMENTO DISTRIBUTIVO (IBOV)")
print("========================================================================\n")
print(f"ARCH-LM (10 lags):  estat = {arch_stat:.2f}  p-valor = {fmt_pvalor(arch_p)}")
print(f'Jarque-Bera:        estat = {jb_stat:.2f}  p-valor = {fmt_pvalor(jb_p)}')
print(f"Assimetria:         {r.skew():+.4f}")
print(f"Curtose excedente:  {r.kurt():+.4f}")

## 5. Teste de Razão de Variâncias de Lo-MacKinlay

In [ ]:
log_price = np.log(p).values
vr_2 = VarianceRatio(log_price, 2, overlap=True)
vr_16 = VarianceRatio(log_price, 16, overlap=True)

print("========================================================================")
print("TESTE DE RAZÃO DE VARIÂNCIAS DE LO-MACKINLAY (1988) DO IBOV")
print("========================================================================\n")
print(f"Razão de Variâncias (k=2):   estat={vr_2.stat:.4f} | p-valor={fmt_pvalor(vr_2.pvalue)}")
print(f"Razão de Variâncias (k=16):  estat={vr_16.stat:.4f} | p-valor={fmt_pvalor(vr_16.pvalue)}")

## 6. Modelagem de Volatilidade Condicional e Parâmetro Delta

In [ ]:
r_pct = r * 100
res_garch = arch_model(r_pct, mean="Constant", vol="Garch", p=1, q=1, dist="normal").fit(disp="off")
res_gjr = arch_model(r_pct, mean="Constant", vol="Garch", p=1, o=1, q=1, dist="t").fit(disp="off")
res_eg = arch_model(r_pct, mean="Constant", vol="EGARCH", p=1, o=1, q=1, dist="t").fit(disp="off")

comp = pd.DataFrame([
    diagnostico_residuos(res_garch, "GARCH(1,1) Normal"),
    diagnostico_residuos(res_gjr,   "GJR-GARCH(1,1,1) t"),
    diagnostico_residuos(res_eg,    "EGARCH(1,1,1) t"),
])

melhor = comp.loc[comp["BIC"].idxmin(), "Modelo"]
res_melhor = {"GARCH(1,1) Normal": res_garch, "GJR-GARCH(1,1,1) t": res_gjr, "EGARCH(1,1,1) t": res_eg}[melhor]

print("========================================================================")
print("COMPARAÇÃO DE MODELOS DE VOLATILIDADE CONDICIONAL E CRITÉRIOS")
print("========================================================================\n")
print(comp.round(4).to_string(index=False))
print(f"\n>>> Modelo selecionado pelo BIC: {melhor}")

## 7. Consolidação de Estratégias e Análise de Sensibilidade

In [ ]:
strat_pq = INPUT_DIR_STRATEGY / "strategy_returns.parquet"
strat_csv = INPUT_DIR_STRATEGY / "strategy_returns.csv"
if strat_pq.exists():
    strat = pd.read_parquet(strat_pq)
else:
    strat = pd.read_csv(strat_csv, index_col=0, parse_dates=True)

df = pd.DataFrame(index=strat.index)
df["ret_ibov"] = ibov.set_index("data")["ibov_ret_simples"].reindex(df.index).ffill().bfill()
df["rf"] = ibov.set_index("data")["cdi_diario"].reindex(df.index).ffill().bfill()

for est in ESTRATEGIAS:
    df[est] = strat[est]
    df[f"excess_{est}"] = df[est] - df["rf"]
df["excess_ibov"] = df["ret_ibov"] - df["rf"]

colunas_teste = ESTRATEGIAS + ["ret_ibov"]
linhas_perf = []
for c in colunas_teste:
    nm = "IBOVESPA" if c == "ret_ibov" else c
    r_serie = df[c]
    sh = sharpe(r_serie, df["rf"])
    so = sortino(r_serie, df["rf"])
    linhas_perf.append({
        "Estratégia": nm,
        "Retorno Acumulado (%)": float((1 + r_serie).prod() - 1) * 100,
        "CAGR (% a.a.)": cagr(r_serie) * 100,
        "Volatilidade (% a.a.)": float(r_serie.std() * np.sqrt(TRADING_DAYS)) * 100,
        "Sharpe": sh,
        "Sortino": so,
        "Max Drawdown (%)": max_drawdown(r_serie) * 100
    })
painel = pd.DataFrame(linhas_perf)

print("========================================================================")
print("PAINEL COMPARATIVO GERAL DE PERFORMANCE E RISCO")
print("========================================================================\n")
print(painel.round(4).to_string(index=False))

## 8. Validação Não-Paramétrica via Stationary Bootstrap

In [ ]:
print(f"Stationary bootstrap (B={BOOTSTRAP_REPS}, block_mean={BOOTSTRAP_BLOCK_MEAN}d)\n")
print(f"{'Métrica Estatística':<22} | {'Estratégia (IC 95%)':<36} | {'IBOVESPA (IC 95%)'}")
print("-" * 100)

for est in ESTRATEGIAS:
    print(f"\n========================================================================")
    print(f"INTERVALOS DE CONFIANÇA POR REAMOSTRAGEM: {est} VS IBOVESPA")
    print("========================================================================")
    for nome_fn, fn, serie_a, serie_b in [
        ("Sharpe (excesso)",  sharpe_de_excesso,  df[f"excess_{est}"], df["excess_ibov"]),
        ("Sortino (excesso)", sortino_de_excesso, df[f"excess_{est}"], df["excess_ibov"]),
        ("CAGR (bruto)",      cagr,               df[est],             df["ret_ibov"]),
    ]:
        s_lo, s_md, s_hi = bootstrap_ic(serie_a, fn, B=BOOTSTRAP_REPS, block_mean=BOOTSTRAP_BLOCK_MEAN, seed=SEED)
        i_lo, i_md, i_hi = bootstrap_ic(serie_b, fn, B=BOOTSTRAP_REPS, block_mean=BOOTSTRAP_BLOCK_MEAN, seed=SEED)
        print(f"{nome_fn:<22} | [{s_lo:+.4f},  {s_md:+.4f},  {s_hi:+.4f}]  | "
              f"[{i_lo:+.4f},  {i_md:+.4f},  {i_hi:+.4f}]")

## 9. CAPM HAC, Jobson-Korkie/Memmel e Wald Spanning

In [ ]:
X_capm = sm.add_constant((df["ret_ibov"] - df["rf"]).values)
X_span = sm.add_constant(df["ret_ibov"].values)
exc_ibov = df["ret_ibov"] - df["rf"]

linhas_testes = []
for est in ESTRATEGIAS:
    exc = df[est] - df["rf"]
    capm = sm.OLS(exc.values, X_capm).fit(cov_type="HAC", cov_kwds={"maxlags": SPANNING_MAX_LAGS})
    z_jk, p_jk = _jk_memmel(exc, exc_ibov)
    _, f_sp, p_sp = _wald_spanning(df[est].values, X_span, maxlags=SPANNING_MAX_LAGS)
    linhas_testes.append({
        "Estrategia":         est,
        "CAPM alfa (% a.a.)": capm.params[0] * TRADING_DAYS * 100,
        "CAPM beta":          capm.params[1],
        "CAPM t_alfa (NW)":   capm.tvalues[0],
        "CAPM p_alfa":        capm.pvalues[0],
        "JK/Memmel z":        z_jk,
        "JK/Memmel p":        p_jk,
        "Spanning F":         f_sp,
        "Spanning p":         p_sp,
        "Sharpe (rf var)":    sharpe(df[est], df["rf"]),
        "Sortino (rf var)":   sortino(df[est], df["rf"]),
    })

apendice_H = pd.DataFrame(linhas_testes).set_index("Estrategia")

print("========================================================================")
print("APÊNDICE L — INFERÊNCIA COMPARATIVA TRANSVERSAL (HAC Newey-West)")
print("========================================================================\n")
for est, row in apendice_H.iterrows():
    print(f"\n> {est}")
    print(f"   CAPM:     alfa = {row['CAPM alfa (% a.a.)']:+.2f}% a.a. | beta = {row['CAPM beta']:.3f} "
          f"| t(NW) = {row['CAPM t_alfa (NW)']:+.3f} | p = {fmt_pvalor(row['CAPM p_alfa'])}")
    print(f"   JK/Memmel vs IBOV: z = {row['JK/Memmel z']:+.3f} | p = {fmt_pvalor(row['JK/Memmel p'])}")
    print(f"   Spanning (a=0,b=1): F = {row['Spanning F']:.2f} | p = {fmt_pvalor(row['Spanning p'])}")
    print(f"   Sharpe = {row['Sharpe (rf var)']:+.4f} | Sortino = {row['Sortino (rf var)']:+.4f}")

## 10. Exportação dos Resultados e Apêndices

In [ ]:
apendice_G = pd.DataFrame([
    {"Teste": "ADF (log-retornos)",     "H₀": "raiz unitária",    "Estat.": adf_r[0],  "p-valor": adf_r[1]},
    {"Teste": "KPSS (log-retornos)",    "H₀": "estacionariedade", "Estat.": kpss_r[0], "p-valor": kpss_r[1]},
    {"Teste": "ADF (nível)",            "H₀": "raiz unitária",    "Estat.": adf_p[0],  "p-valor": adf_p[1]},
    {"Teste": "KPSS (nível)",           "H₀": "estacionariedade", "Estat.": kpss_p[0], "p-valor": kpss_p[1]},
    {"Teste": "ARCH-LM (10 lags)",      "H₀": "variância constante", "Estat.": arch_stat, "p-valor": arch_p},
    {"Teste": "Jarque-Bera",            "H₀": "normalidade",      "Estat.": jb_stat,        "p-valor": jb_p},
    {"Teste": "VR k=2 (Lo-MacKinlay)",  "H₀": "passeio aleatório", "Estat.": vr_2.stat,  "p-valor": vr_2.pvalue},
    {"Teste": "VR k=16 (Lo-MacKinlay)", "H₀": "passeio aleatório", "Estat.": vr_16.stat, "p-valor": vr_16.pvalue},
])
apendice_G["Decisão"] = np.where(apendice_G["p-valor"] < 0.05, "Rejeita H₀ ✗", "Não rejeita H₀ ✓")
apendice_G.to_csv(OUTPUT_DIR / "apendice_G_diagnostico_ibov.csv", index=False)

apendice_H.to_csv(OUTPUT_DIR / "apendice_H_testes_estrategia.csv", float_format="%.6f")
painel.to_csv(OUTPUT_DIR / "apendice_H_painel_metricas.csv", index=False)

comp_rf_linhas = []
for c in colunas_teste:
    nm = "IBOVESPA" if c == "ret_ibov" else c
    comp_rf_linhas.append({
        "Estratégia": nm,
        "Sharpe (CDI)": sharpe(df[c], df["rf"]),
        "Sharpe (Zero)": sharpe(df[c], 0.0),
        "Sortino (CDI)": sortino(df[c], df["rf"]),
        "Sortino (Zero)": sortino(df[c], 0.0),
    })
comp_rf = pd.DataFrame(comp_rf_linhas)
comp_rf.to_csv(OUTPUT_DIR / "apendice_H_comparativo_rf.csv", index=False)

print("✓ CSVs de apêndice exportados com sucesso em:", OUTPUT_DIR)